# Neo4j Simple Connection Test

Validates connectivity and creates the UC JDBC connection.

**Prerequisites:**
- Run `./getting-started/upload_data.sh` (from the repo root) to upload CSV files to the UC Volume
- Run `./create_secrets.sh` (from the repo root) to store notebook configuration as Databricks secrets
- Run `00-load-graph.ipynb` to load the aircraft graph into Neo4j

## Graph Schema

This notebook validates the UC JDBC connection by running simple `remote_query()` calls against three node labels.

<table>
<tr><td>

**Nodes**

| Label | Properties |
|---|---|
| Aircraft | aircraftId, tail_number, icao24, model, manufacturer, operator |
| Airport | airportId, name, city, country, iata, icao |
| System | systemId, aircraftId, type, name |
| Component | componentId, systemId, type, name |
| Sensor | sensorId, systemId, type, name, unit |
| Flight | flightId, flight_number, aircraftId, operator, origin, destination, scheduled_departure, scheduled_arrival |
| MaintenanceEvent | eventId, componentId, systemId, aircraftId, fault, severity, reported_at, corrective_action |
| Delay | delayId, flightId, cause, minutes |

</td><td>

**Relationships**

| Type | From → To |
|---|---|
| HAS_SYSTEM | Aircraft → System |
| HAS_COMPONENT | System → Component |
| HAS_SENSOR | System → Sensor |
| OPERATES_FLIGHT | Aircraft → Flight |
| DEPARTS_FROM | Flight → Airport |
| ARRIVES_AT | Flight → Airport |
| HAS_DELAY | Flight → Delay |
| HAS_EVENT | Component → MaintenanceEvent |

</td></tr>
</table>

## About `remote_query()`

This is the first time the project uses `remote_query()`, the Databricks SQL table-valued function that lets a single SQL statement query a foreign source through a UC JDBC connection:

```sql
SELECT * FROM remote_query('<uc_connection_name>',
    query => '<SQL string sent to Neo4j>'
)
```

Databricks sends the inner SQL string verbatim through the UC connection. On the Neo4j side, the JDBC driver's SQL-to-Cypher translator parses the SQL and emits Cypher. Notebook 02 builds on this to do cross-source single-statement joins with Delta tables.

### Why the `HAVING COUNT(*) > 0` workaround

`remote_query()` has a result-reuse bug that can return an empty result set for pure `GROUP BY` queries on warm clusters (second and later runs in the same session). Adding `HAVING COUNT(*) > 0` is semantically a no-op — every aggregated group has at least one row by definition — but it forces the planner to re-evaluate and bypasses the empty cache. The bug is in Databricks' `remote_query()`, not in the Neo4j JDBC driver. Documented in `site/modules/ROOT/pages/troubleshooting.adoc`. You'll see it on the "flights by operator" query in Section 2.

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Loaded from Databricks secrets
# =============================================================================

# --- Neo4j Aura ---
SECRET_SCOPE = "neo4j-uc-demos"
NEO4J_URI = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_URI")
NEO4J_USERNAME = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_USERNAME")
NEO4J_PASSWORD = dbutils.secrets.get(scope=SECRET_SCOPE, key="NEO4J_PASSWORD")

# --- Databricks Unity Catalog ---
UC_CATALOG = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_CATALOG")
UC_SCHEMA = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_SCHEMA")
UC_VOLUME = dbutils.secrets.get(scope=SECRET_SCOPE, key="UC_VOLUME")
JDBC_JAR_PATH = dbutils.secrets.get(scope=SECRET_SCOPE, key="JDBC_JAR_PATH")
UC_CONNECTION_NAME = "sample_neo4j_jdbc_connection"

# =============================================================================
# DERIVED VALUES - no need to edit below this line
# =============================================================================
FQN = f"`{UC_CATALOG}`.`{UC_SCHEMA}`"
VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"
NEO4J_JDBC_URL_SQL = (
    f"jdbc:{NEO4J_URI}/neo4j"
    "?enableSQLTranslation=true"
    "&timeout=30000"
)
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'

print("Configuration:")
print(f"  Neo4j URI:       {NEO4J_URI}")
print(f"  Tables:          {FQN}.*")
print(f"  Volume:          {VOLUME_PATH}")
print(f"  JDBC JAR:        {JDBC_JAR_PATH}")
print(f"  UC Connection:   {UC_CONNECTION_NAME}")

---

## Section 1: Create UC JDBC Connection

Creates the Unity Catalog JDBC connection that downstream notebooks use for federation, then validates it with a trivial `remote_query()` call.

**Note:** `java_dependencies` only accepts UC Volume paths. The SafeSpark sandbox isolates the JDBC driver in its own JVM, which is why the project README documents the metaspace tuning requirement.

In [ ]:
import time

print("--- Section 1: Create UC JDBC Connection ---")
print(f"  Connection: {UC_CONNECTION_NAME}")
print(f"  URL:        {NEO4J_JDBC_URL_SQL}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

esc = lambda s: s.replace("'", "\\'")

create_sql = f"""
    CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
    ENVIRONMENT (
        java_dependencies '{JAVA_DEPENDENCIES}'
    )
    OPTIONS (
        url '{esc(NEO4J_JDBC_URL_SQL)}',
        user '{esc(NEO4J_USERNAME)}',
        password '{esc(NEO4J_PASSWORD)}',
        driver 'org.neo4j.jdbc.Neo4jDriver',
        externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
    )
"""

start = time.time()
spark.sql(create_sql)
elapsed = (time.time() - start) * 1000
print(f"  [PASS] Connection created in {elapsed:.0f}ms")

# Validate the connection with a trivial remote_query() call.
test_val = spark.sql(f"""
    SELECT test FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT 1 AS test'
    )
""").collect()[0]["test"]
status = "PASS" if test_val == 1 else "FAIL"
print(f"  [{status}] remote_query() returned {test_val}")

---

## Section 2: Query Neo4j via `remote_query()`

Three `remote_query()` calls against the graph: two `COUNT(*)` queries on node labels, then a `GROUP BY` aggregate that shows aggregation being pushed into Neo4j (only summarized rows cross the wire).

The `HAVING COUNT(*) > 0` clause on the third query is the result-reuse workaround described in the note near the top of the notebook.

**SQL → Cypher examples:**

| SQL | Cypher |
|-----|--------|
| `SELECT COUNT(*) AS aircraft_count FROM Aircraft` | `MATCH (aircraft:Aircraft) RETURN count(*) AS aircraft_count` |
| `SELECT COUNT(*) AS airport_count FROM Airport` | `MATCH (airport:Airport) RETURN count(*) AS airport_count` |
| `SELECT f.operator AS operator, COUNT(*) AS flight_count FROM Flight f GROUP BY f.operator HAVING COUNT(*) > 0` | `MATCH (f:Flight) WITH f.operator AS operator, count(*) AS flight_count WHERE flight_count > 0 RETURN operator, flight_count` |

In [ ]:
print("--- Section 2: Query Neo4j via remote_query() ---")

# Aircraft count
count = spark.sql(f"""
    SELECT aircraft_count FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS aircraft_count FROM Aircraft'
    )
""").collect()[0]["aircraft_count"]
status = "PASS" if count == 20 else "FAIL"
print(f"  [{status}] Aircraft: {count}")

# Airport count
count = spark.sql(f"""
    SELECT airport_count FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT COUNT(*) AS airport_count FROM Airport'
    )
""").collect()[0]["airport_count"]
status = "PASS" if count == 12 else "FAIL"
print(f"  [{status}] Airports: {count}")

# Flights by operator (aggregate pushdown — Neo4j does the GROUP BY).
# HAVING COUNT(*) > 0 is the remote_query() GROUP BY workaround (see note at top).
print("\n  Flights by operator:")
spark.sql(f"""
    SELECT operator, flight_count FROM remote_query('{UC_CONNECTION_NAME}',
        query => 'SELECT f.operator AS operator, COUNT(*) AS flight_count
                  FROM Flight f
                  GROUP BY f.operator
                  HAVING COUNT(*) > 0'
    )
    ORDER BY flight_count DESC
""").show(truncate=False)

print("Status: PASS. Run 02-federated-queries.ipynb next")